In [1]:
import pandas as pd

In [2]:
df_ori = pd.read_csv("GoogleReview_data_mix.csv")

In [3]:
df_ori.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85513 entries, 0 to 85512
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Author      85513 non-null  object
 1   Rating      85513 non-null  int64 
 2   Review      85513 non-null  object
 3   Restaurant  85513 non-null  object
 4   Location    85512 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.3+ MB


In [4]:
# Adding labels
def labelFunc(x):
    if int(x) > 3:
        return "Positive"
    elif int(x) < 3:
        return "Neutral"
    elif int(x) == 3:
        return "Negative"

# Add sentiment label
df_ori["Label"] = df_ori["Rating"].apply(labelFunc)

In [5]:
df_ori.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85513 entries, 0 to 85512
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Author      85513 non-null  object
 1   Rating      85513 non-null  int64 
 2   Review      85513 non-null  object
 3   Restaurant  85513 non-null  object
 4   Location    85512 non-null  object
 5   Label       85513 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.9+ MB


In [6]:
from sklearn.model_selection import train_test_split

#Separate the final Test Set (20%)
df_temp, df_test = train_test_split(
    df_ori,
    test_size=0.20,
    random_state=42,
    stratify=df_ori['Label'] # Keeps class balance stable
)

#Split the 80% temporary pool into Train (60%) and Val (20%) ---
df_train, df_val = train_test_split(
    df_temp,
    test_size=0.25,
    random_state=42,
    stratify=df_temp['Label']
)

# --- VERIFY THE SPLITS ---
print(f"Total Rows in Dataset: {len(df_ori)}")
print(f"Training Set (60%):   {len(df_train)} rows")
print(f"Validation Set (20%): {len(df_val)} rows")
print(f"Testing Set (20%):    {len(df_test)} rows")

print("\nVerify Stratification (Should be nearly identical percentages):")
print("Train:\n", df_train['Label'].value_counts(normalize=True))
print("Val:\n", df_val['Label'].value_counts(normalize=True))
print("Test:\n", df_test['Label'].value_counts(normalize=True))

Total Rows in Dataset: 85513
Training Set (60%):   51307 rows
Validation Set (20%): 17103 rows
Testing Set (20%):    17103 rows

Verify Stratification (Should be nearly identical percentages):
Train:
 Label
Positive    0.778666
Negative    0.121348
Neutral     0.099986
Name: proportion, dtype: float64
Val:
 Label
Positive    0.778635
Negative    0.121382
Neutral     0.099982
Name: proportion, dtype: float64
Test:
 Label
Positive    0.778694
Negative    0.121324
Neutral     0.099982
Name: proportion, dtype: float64


In [7]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51307 entries, 37237 to 72088
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Author      51307 non-null  object
 1   Rating      51307 non-null  int64 
 2   Review      51307 non-null  object
 3   Restaurant  51307 non-null  object
 4   Location    51307 non-null  object
 5   Label       51307 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.7+ MB


## using huggingface model

In [17]:
import torch
from transformers import pipeline
from transformers import AutoTokenizer, T5ForConditionalGeneration

### train dataset


In [14]:
# Create a clean copy to modify in-place
df_train_bilingual = df_train.copy()
df_train_bilingual['Language'] = 'English'  # Set default label

# Extract the unique row indices representing exactly 30% of the data
malay_indices = df_train_bilingual.sample(frac=0.30, random_state=42).index

print(f"Total rows in df_train: {len(df_train_bilingual)}")
print(f"Selecting {len(malay_indices)} rows to turn into Malay...")

# Detect hardware: automatically use GPU (device=0) if available, fallback to CPU (device=-1)
device = 0 if torch.cuda.is_available() else -1
print(f"Loading Hugging Face model on: {'GPU (CUDA)' if device == 0 else 'CPU'}")

Total rows in df_train: 51307
Selecting 15392 rows to turn into Malay...
Loading Hugging Face model on: GPU (CUDA)


In [19]:
# Initialize the Hugging Face translation pipeline
# Load tokenizer and model directly to bypass the pipeline task registry
model_name = "mesolitica/nanot5-small-malaysian-translation-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

# Prepare text list with the mandatory model prompt prefix
reviews_to_translate = [
    f"terjemah ke Melayu: {str(review)}"
    for review in df_train_bilingual.loc[malay_indices, 'Review']
]

print("Running batch direct processing loop...")
translated_texts = []
batch_size = 64  # Lower to 32 or 16 if your computer runs out of memory (OOM)

for i in range(0, len(reviews_to_translate), batch_size):
    batch = reviews_to_translate[i:i + batch_size]

    # Tokenize input text array
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

    # Generate translated tokens matrix
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=512)

    # Decode matrix back to standard text strings
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    translated_texts.extend(decoded)

# Overwrite original reviews with clean string conversions in-place
df_train_bilingual.loc[malay_indices, 'Review'] = translated_texts
df_train_bilingual.loc[malay_indices, 'Language'] = 'Malay'

print("Translation Complete!")

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Running batch direct processing loop...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Translation Complete!


In [20]:
df_train_bilingual.head(10)

,Author,Rating,Review,Restaurant,Location,Label,Language
37237,wu stephen,5,Good and fresh food,EverGreen Restaurant,JB,Positive,English
58570,Nicole Lim,4,"Kami makan ponteh, ikan Assam, sotong kunyit,...",Restoran Nyonya Makko,Melaka,Positive,Malay
28309,Mohd Khairuddin Jaafar,5,Nice meal,Restoran Kacang Pool Haji,JB,Positive,English
33177,sze yah chng,3,"The curry taste more to like Johor type, more ...",W.W. Laksa House 水塘。辣沙,JB,Negative,English
44121,Sh Lim,4,"Satu, yang terbaik adalah JB Bak Kut Teh. Suk...",Restaurant Shun Fa Bak Kut Teh,JB,Positive,Malay
39731,Thanabalan Vadivelu,2,I like the food,KAMPUNG CARABAO 100% Authentic Thai Restaurant,JB,Neutral,English
38019,Li Min Leong,4,Sashimi segar. Semua rasa YUMS.,Yaoki Japanese Foods Sdn. Bhd.,JB,Positive,Malay
52549,Kim de Buck,5,Sebuah restoran kecil yang sangat selesa deng...,Trois by navy,Melaka,Positive,Malay
38560,Robert Choong,4,Nice place 👍,Harvests Bar & Grill,JB,Positive,English
26718,mimie aqilah mohd amran,4,Keep coming here for good Indian cuisine. staf...,7 Spice,JB,Positive,English


In [21]:
df_train_bilingual.to_csv('train_dataset_bilingual.csv', index=False)

### validate dataset

In [22]:
# Create a clean copy to modify in-place
df_val_bilingual = df_val.copy()
df_val_bilingual['Language'] = 'English'  # Set default label

# Extract the unique row indices representing exactly 30% of the data
malay_indices = df_val_bilingual.sample(frac=0.30, random_state=42).index

print(f"Total rows in df_val: {len(df_val_bilingual)}")
print(f"Selecting {len(malay_indices)} rows to turn into Malay...")

# Detect hardware: automatically use GPU (device=0) if available, fallback to CPU (device=-1)
device = 0 if torch.cuda.is_available() else -1
print(f"Loading Hugging Face model on: {'GPU (CUDA)' if device == 0 else 'CPU'}")

Total rows in df_train: 17103
Selecting 5131 rows to turn into Malay...
Loading Hugging Face model on: GPU (CUDA)


In [23]:
# Initialize the Hugging Face translation pipeline
# Load tokenizer and model directly to bypass the pipeline task registry
model_name = "mesolitica/nanot5-small-malaysian-translation-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

# Prepare text list with the mandatory model prompt prefix
reviews_to_translate = [
    f"terjemah ke Melayu: {str(review)}"
    for review in df_val_bilingual.loc[malay_indices, 'Review']
]

print("Running batch direct processing loop...")
translated_texts = []
batch_size = 64  # Lower to 32 or 16 if your computer runs out of memory (OOM)

for i in range(0, len(reviews_to_translate), batch_size):
    batch = reviews_to_translate[i:i + batch_size]

    # Tokenize input text array
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

    # Generate translated tokens matrix
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=512)

    # Decode matrix back to standard text strings
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    translated_texts.extend(decoded)

# Overwrite original reviews with clean string conversions in-place
df_val_bilingual.loc[malay_indices, 'Review'] = translated_texts
df_val_bilingual.loc[malay_indices, 'Language'] = 'Malay'

print("Translation Complete!")

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Running batch direct processing loop...
Translation Complete!


In [24]:
df_val_bilingual.head(10)

,Author,Rating,Review,Restaurant,Location,Label,Language
56560,S.Y. Ding,5,"Popiah yang bagus, saya suka popiah. Saya san...",Poh Piah Lwee,Melaka,Positive,Malay
5982,Ken Chong,4,A must try when you're in Ipoh. we had the fis...,Ipoh Tuck Kee Restaurant 德记炒粉专门店,Ipoh,Positive,English
13855,H.,3,Nothing really special,Foh San Restaurant,Ipoh,Negative,English
51278,Jo huang,5,Salah satu tempat makan malam terbaik di Pela...,海皇粿条仔Restaurant Hi Wan,JB,Positive,Malay
34300,zaeim ismail,3,been there couple mnths back. visited again to...,K Fry Urban Korean Holiday Villa,JB,Negative,English
5837,Esther G. Jayaraja,4,Mee goreng yang hebat! Hor fun moonlight sang...,Ipoh Tuck Kee Restaurant 德记炒粉专门店,Ipoh,Positive,Malay
2291,Dzin Norain,5,"the Dim Sum is nice n delicious , portion is i...",Ming Court Hong Kong Dim Sum 明阁香港点心,Ipoh,Positive,English
16187,Nedu Chalian,1,Shame 2 say the food is very bad the wrost ind...,Anderson Curry House,Ipoh,Neutral,English
53028,SALLY,5,Hanya hidangan burger dengan kentang goreng d...,The Baboon House,Melaka,Positive,Malay
21048,wenny Alimodo,5,"When we think about celebrations, Wengkee is t...",Weng Kee Seafood Restaurant（永記海鮮飯店）,Ipoh,Positive,English


In [25]:
df_val_bilingual.to_csv('validate_dataset_bilingual.csv', index=False)

### test dataset

In [26]:
# Create a clean copy to modify in-place
df_test_bilingual = df_test.copy()
df_test_bilingual['Language'] = 'English'  # Set default label

# Extract the unique row indices representing exactly 30% of the data
malay_indices = df_test_bilingual.sample(frac=0.30, random_state=42).index

print(f"Total rows in df_test: {len(df_test_bilingual)}")
print(f"Selecting {len(malay_indices)} rows to turn into Malay...")

# Detect hardware: automatically use GPU (device=0) if available, fallback to CPU (device=-1)
device = 0 if torch.cuda.is_available() else -1
print(f"Loading Hugging Face model on: {'GPU (CUDA)' if device == 0 else 'CPU'}")

Total rows in df_test: 17103
Selecting 5131 rows to turn into Malay...
Loading Hugging Face model on: GPU (CUDA)


In [27]:
# Initialize the Hugging Face translation pipeline
# Load tokenizer and model directly to bypass the pipeline task registry
model_name = "mesolitica/nanot5-small-malaysian-translation-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

# Prepare text list with the mandatory model prompt prefix
reviews_to_translate = [
    f"terjemah ke Melayu: {str(review)}"
    for review in df_test_bilingual.loc[malay_indices, 'Review']
]

print("Running batch direct processing loop...")
translated_texts = []
batch_size = 64  # Lower to 32 or 16 if your computer runs out of memory (OOM)

for i in range(0, len(reviews_to_translate), batch_size):
    batch = reviews_to_translate[i:i + batch_size]

    # Tokenize input text array
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

    # Generate translated tokens matrix
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=512)

    # Decode matrix back to standard text strings
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    translated_texts.extend(decoded)

# Overwrite original reviews with clean string conversions in-place
df_test_bilingual.loc[malay_indices, 'Review'] = translated_texts
df_test_bilingual.loc[malay_indices, 'Language'] = 'Malay'

print("Translation Complete!")

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Running batch direct processing loop...
Translation Complete!


In [28]:
df_test_bilingual.head(10)


,Author,Rating,Review,Restaurant,Location,Label,Language
80087,Philomina Wilson,5,Saya suka pastri mereka. Salah satu yang terb...,Chef At Home,Kuching,Positive,Malay
36271,Michael Cheng,5,A very enjoyable brunch in a beautiful cafe & ...,Tropique Café & Restaurant,JB,Positive,English
80013,Keai Aishiteru,5,Beef noodles rm6 comfortable blh hold juak Ras...,Satok Fly Over Cafe,Kuching,Positive,English
70519,michael ting,5,Tempat terbaik untuk mi segera. Tempat terbai...,Kim Joo 錦裕,Kuching,Positive,Malay
23814,Tirath Khera,5,"I love the food here, especially the parantha ...",Restoran Moga Punjab,Ipoh,Positive,English
60469,Stephanie Ong,5,"Suka Prawn Masak Lemak, campuran Pong Teh dan...",SamFu Restaurant,Melaka,Positive,Malay
13948,Yip Fung Lee,1,I don’t know what Hong Kong-style snacks are. ...,Foh San Restaurant,Ipoh,Neutral,English
47682,Treemore Plant,4,Try the potato noodle,Korean BBQ Nam Moon,JB,Positive,English
8792,Firdous Ah Khan,5,Kehormatan makanan. Syurga untuk kekasih buru...,Pakeeza Restaurant & Catering,Ipoh,Positive,Malay
6205,Stephen Philip,5,Food is very good,Ipoh Tuck Kee Restaurant 德记炒粉专门店,Ipoh,Positive,English


In [30]:
#check percentage of malay in each df
print(df_train_bilingual['Language'].value_counts(normalize=True))
print(df_val_bilingual['Language'].value_counts(normalize=True))
print(df_test_bilingual['Language'].value_counts(normalize=True))

Language
English    0.700002
Malay      0.299998
Name: proportion, dtype: float64
Language
English    0.699994
Malay      0.300006
Name: proportion, dtype: float64
Language
English    0.699994
Malay      0.300006
Name: proportion, dtype: float64


In [29]:
df_test_bilingual.to_csv('test_dataset_bilingual.csv', index=False)
